In [ ]:
# CROSS-SILO FL — CELL 1: setup, reuse check, output folder, Kaggle auth, CICIDS2017 mirror listing (CPU, ~1-2 min)
import os, glob, json, subprocess, shutil, sys
import numpy as np, pandas as pd
from google.colab import drive, userdata
drive.mount('/content/drive')

# ---- 1) find the thesis project folder (same rule as before; never print paths) ----
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1, f'expected one project folder, found {len(cands)} - STOP'
DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'
XS = f'{DRIVE_PATH}/results/crosssilo'; XSD = f'{D}/crosssilo'
os.makedirs(XS, exist_ok=True); os.makedirs(XSD, exist_ok=True)
print('Project folder found; new output folders ready: results/crosssilo, data/crosssilo')

# ---- 2) reuse check: the leak-free CIC IoT-DIAD assets and the training harness ----
need = ['fl_core.py', 'split_random.npz', 'split_time.npz']
for n in need:
    print(f'  data/v2/{n:<18} {"OK" if os.path.exists(f"{V2D}/{n}") else "MISSING"}')
assert all(os.path.exists(f'{V2D}/{n}') for n in need), 'a required file is missing - STOP and tell Claude'
src = open(f'{V2D}/fl_core.py').read()
print('  fl_core.py version markers:', {k: (k in src) for k in ['control variates track the CLIPPED', "'pred_test': last_pred", 'local_epochs', 'fednova']})
for sp in ('random', 'time'):
    S = np.load(f'{V2D}/split_{sp}.npz', allow_pickle=True)
    print(f'  split_{sp}: keys {sorted(S.files)}')
    print(f'     X_train {S["X_train"].shape}  X_val {S["X_val"].shape}  X_test {S["X_test"].shape}  classes {np.unique(S["y_train"]).tolist()}')
print('  other files in data/v2:', sorted(os.path.basename(p) for p in glob.glob(f'{V2D}/*'))[:40])
print('  CIC column file present (data/X_cleaned.parquet):', os.path.exists(f'{D}/X_cleaned.parquet'))
if os.path.exists(f'{D}/X_cleaned.parquet'):
    cols = list(pd.read_parquet(f'{D}/X_cleaned.parquet').columns)
    print(f'  X_cleaned columns: {len(cols)}')
print('  earlier CICIDS2017 file on Drive:', [os.path.basename(p) for p in glob.glob(f'{D}/*cicids2017*')])

# ---- 3) machine check ----
import psutil
print(f'\nRAM {psutil.virtual_memory().total / 1e9:.1f} GB | free disk {shutil.disk_usage("/content").free / 1e9:.1f} GB')
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'none (fine for this cell)')

# ---- 4) Kaggle auth from Colab secrets (token is never printed) ----
token = userdata.get('KAGGLE_API_TOKEN')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
tp = os.path.expanduser('~/.kaggle/access_token')
open(tp, 'w').write(token); os.chmod(tp, 0o600); del token
r = subprocess.run(['kaggle', 'datasets', 'files', '-d', 'shadman1028/cicids2017-official-flow-feature-csv-files'],
                   capture_output=True, text=True)
print('\nKaggle auth', 'OK' if r.returncode == 0 else 'FAILED')
print('--- CICIDS2017 mirror (official flow-feature CSVs) ---')
print(r.stdout[-2500:] if r.returncode == 0 else r.stderr[-800:])

# ---- 5) look for a corrected/relabelled CICIDS2017 on Kaggle (listing only, nothing downloaded) ----
for q in ['cicids2017 improved', 'cic-ids2017 corrected', 'cicids2017 relabelled']:
    r2 = subprocess.run(['kaggle', 'datasets', 'list', '-s', q], capture_output=True, text=True)
    print(f'\n--- search: "{q}" ---'); print(r2.stdout[:1200] if r2.returncode == 0 else r2.stderr[:300])

In [ ]:
# CROSS-SILO FL — CELL 2 (v2): inspect existing CICIDS2017 assets + the Improved CICIDS2017 release (CPU, ~3-6 min)
import os, glob, json, subprocess, zipfile, joblib
import numpy as np, pandas as pd
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'

# ---- 1) the 68 feature names used by the leak-free splits (saved with joblib in Phase 2, Cell 1) ----
P = {sp: joblib.load(f'{V2D}/prep_{sp}.pkl') for sp in ('random', 'time')}
for sp, p in P.items():
    print(f'prep_{sp}: keys {list(p.keys())} | n columns {len(p["columns"])} | classes {p["classes"]}')
same = P['random']['columns'] == P['time']['columns']
print('random and time splits use the same 68 columns in the same order:', same)
if not same:
    print('  only in random:', sorted(set(P['random']['columns']) - set(P['time']['columns'])))
    print('  only in time  :', sorted(set(P['time']['columns']) - set(P['random']['columns'])))
CIC68 = P['random']['columns']
cic_cols = list(pd.read_parquet(f'{D}/X_cleaned.parquet').columns)
print('same set as the old X_cleaned.parquet columns:', set(cic_cols) == set(CIC68))
print('CIC68 first 8:', CIC68[:8])

# ---- 2) existing alignment file and the earlier full CICIDS2017 download ----
al = json.load(open(f'{D}/cicids2017_alignment.json'))
print('\nalignment json keys:', list(al.keys()) if isinstance(al, dict) else type(al).__name__)
for k, v in (al.items() if isinstance(al, dict) else []):
    print(f'   {k}: {v if not isinstance(v, (list, dict)) else (len(v), (list(v.items())[:3] if isinstance(v, dict) else v[:3]))}')
full = f'{D}/cicids2017_full'
print('\ncicids2017_full is a', 'folder' if os.path.isdir(full) else 'file')
if os.path.isdir(full):
    for p in sorted(glob.glob(f'{full}/**/*', recursive=True))[:40]:
        if os.path.isfile(p): print(f'   {os.path.relpath(p, full):<70} {os.path.getsize(p) / 1e6:9.1f} MB')

# ---- 3) Improved CICIDS2017 (relabelled release): list files, fetch the smallest CICIDS2017 one ----
REF = 'ernie55ernie/improved-cicids2017-and-csecicids2018'
r = subprocess.run(['kaggle', 'datasets', 'files', '-d', REF, '--page-size', '200'], capture_output=True, text=True)
rows = []
for l in r.stdout.splitlines():
    parts = l.split()
    if len(parts) >= 2 and parts[1].isdigit(): rows.append((parts[0], int(parts[1])))
F = pd.DataFrame(rows, columns=['file', 'bytes'])
print(f'\nImproved release: {len(F)} files listed' + ('' if r.returncode == 0 else f' | ERROR: {r.stderr[-300:]}'))
if len(F): print(F.assign(MB=(F.bytes / 1e6).round(1)).drop(columns='bytes').to_string(index=False))
c17 = F[~F.file.str.contains('2018', case=False)].sort_values('bytes') if len(F) else F
if len(c17):
    pick = c17.iloc[0].file; tmp = '/content/improved_check'; os.makedirs(tmp, exist_ok=True)
    print(f'\nDownloading the smallest CICIDS2017 file for a column check: {pick} ({c17.iloc[0].bytes / 1e6:.1f} MB)')
    rr = subprocess.run(['kaggle', 'datasets', 'download', '-d', REF, '-f', pick, '-p', tmp], capture_output=True, text=True)
    if rr.returncode != 0: print('download failed:', rr.stderr[-500:])
    for z in glob.glob(f'{tmp}/*.zip'):
        zipfile.ZipFile(z).extractall(tmp)
    csvs = glob.glob(f'{tmp}/**/*.csv', recursive=True)
    if csvs:
        s = pd.read_csv(csvs[0], nrows=200000, low_memory=False); s.columns = [c.strip() for c in s.columns]
        print(f'columns ({s.shape[1]}):', list(s.columns))
        for c in [c for c in s.columns if c.lower() in ('label', 'attempted category', 'attempted_category')]:
            print(f'\n{c} counts (first 200k rows):\n{s[c].value_counts().head(20).to_string()}')
        norm = lambda x: ''.join(ch for ch in x.lower() if ch.isalnum())
        imp = {norm(c): c for c in s.columns}
        print(f'\nCIC68 features with a same-name column in the Improved release (case/space-insensitive): '
              f'{sum(norm(c) in imp for c in CIC68)}/68')
        print('not matched by name:', [c for c in CIC68 if norm(c) not in imp])

In [ ]:
# CROSS-SILO FL — CELL 3: download Improved CICIDS2017 (5 day files), keep needed columns, save parquet to Drive,
#                         full-release label counts per day (CPU High-RAM, ~10-20 min; skips work already done)
import os, glob, json, subprocess, zipfile, time, joblib
import numpy as np, pandas as pd
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
REF = 'ernie55ernie/improved-cicids2017-and-csecicids2018'
DAYS = ['monday', 'tuesday', 'wednesday', 'thursday', 'friday']
CIC68 = joblib.load(f'{V2D}/prep_random.pkl')['columns']
ID_COLS = ['id', 'Flow ID', 'Src IP', 'Src Port', 'Dst IP', 'Dst Port', 'Protocol', 'Timestamp']
KEEP_META = ['Src IP', 'Timestamp', 'Label', 'Attempted Category']
OUT = f'{XSD}/cicids2017_improved.parquet'
t0 = time.time()

if not os.path.exists(OUT):
    tmp = '/content/cic17_improved'; os.makedirs(tmp, exist_ok=True)
    parts = []
    for d in DAYS:
        f = f'CICIDS2017_improved/{d}.csv'
        csv = glob.glob(f'{tmp}/**/{d}.csv', recursive=True)
        if not csv:
            r = subprocess.run(['kaggle', 'datasets', 'download', '-d', REF, '-f', f, '-p', tmp], capture_output=True, text=True)
            assert r.returncode == 0, f'download failed for {d}: {r.stderr[-300:]}'
            for z in glob.glob(f'{tmp}/*.zip'):
                zipfile.ZipFile(z).extractall(tmp); os.remove(z)
            csv = glob.glob(f'{tmp}/**/{d}.csv', recursive=True)
        assert len(csv) == 1, f'{d}: csv not found after download - STOP'
        head = pd.read_csv(csv[0], nrows=5); head.columns = [c.strip() for c in head.columns]
        feat_cols = [c for c in head.columns if c not in ID_COLS + ['Label', 'Attempted Category']]
        assert all(c in head.columns for c in CIC68), f'{d}: an aligned feature is missing - STOP'
        df = pd.read_csv(csv[0], low_memory=False); df.columns = [c.strip() for c in df.columns]
        num = df[feat_cols].apply(pd.to_numeric, errors='coerce').astype('float64')
        part = pd.concat([df[KEEP_META].reset_index(drop=True), num.reset_index(drop=True)], axis=1)
        part.insert(0, 'day', d)
        parts.append(part)
        print(f'  {d:<10} rows {len(part):>9,} | numeric non-identifier features {len(feat_cols)} | {(time.time() - t0) / 60:.1f} min', flush=True)
        del df, num
    X = pd.concat(parts, ignore_index=True); del parts
    X['Src IP'] = X['Src IP'].astype(str); X['Timestamp'] = X['Timestamp'].astype(str); X['Label'] = X['Label'].astype(str)
    X.to_parquet(OUT, index=False)
    print(f'saved data/crosssilo/cicids2017_improved.parquet ({os.path.getsize(OUT) / 1e6:.0f} MB)')
else:
    print('parquet already on Drive - loading it')
X = pd.read_parquet(OUT)
feat_cols = [c for c in X.columns if c not in ['day'] + KEEP_META]
print(f'\nImproved CICIDS2017: {len(X):,} rows | {len(feat_cols)} numeric non-identifier features | all 68 aligned features present: '
      f'{all(c in X.columns for c in CIC68)}')

# ---- full-release label counts (before any sampling) ----
X['attempted'] = X['Label'].str.contains('Attempted', case=False)
print(f'\n"Attempted" flows (excluded from the main task): {int(X.attempted.sum()):,}')
tab = pd.crosstab(X['Label'], X['day']).reindex(columns=DAYS, fill_value=0)
tab['total'] = tab.sum(1)
print('\nLabel x day (full release):'); print(tab.sort_values('total', ascending=False).to_string())

# ---- data-quality checks on the 68 aligned features ----
A = X.loc[~X.attempted, CIC68]
print(f'\nNon-finite values in the 68 aligned features (non-attempted rows): '
      f'NaN {int(A.isna().sum().sum()):,} | inf {int(np.isinf(A.to_numpy()).sum()):,}')
print('Distinct source IPs:', X['Src IP'].nunique(), '| timestamp examples:', X['Timestamp'].head(3).tolist())
print(f'\nCELL 3 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSS-SILO FL — CELL 4: label families, sampling, duplicate audit, leak-free splits for BOTH domains,
#                         per-domain train-only preprocessing (CPU High-RAM, ~5-10 min)
import os, glob, json, time, joblib
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
CIC68 = joblib.load(f'{V2D}/prep_random.pkl')['columns']
FAM = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon', 'Web']          # index 0-4 = main task, 5 = Web (sensitivity)
SEED, CAP = 42, 50000
t0 = time.time(); summary = {}

# ====================== ENTERPRISE: Improved CICIDS2017 ======================
E = pd.read_parquet(f'{XSD}/cicids2017_improved.parquet')
E = E[~E['Label'].str.contains('Attempted', case=False)].reset_index(drop=True)
def fam17(l):
    if l == 'BENIGN': return 'Benign'
    if l in ('FTP-Patator', 'SSH-Patator'): return 'Brute Force'
    if l.startswith('DoS '): return 'DoS'
    if l == 'DDoS': return 'DDoS'
    if l == 'Portscan': return 'Recon'
    if l.startswith('Web Attack'): return 'Web'
    return 'OTHER:' + l                                             # Botnet, Infiltration, Infiltration - Portscan, Heartbleed
E['family'] = E['Label'].map(fam17)
print('Enterprise rows by family (non-attempted):'); print(E['family'].value_counts().to_string())
E = E[E['family'].isin(FAM)].reset_index(drop=True)
# stratified sample: cap each family at 50,000, proportional to the original sub-labels inside the family
rng = np.random.default_rng(SEED); keep = []
for f, g in E.groupby('family'):
    if len(g) <= CAP: keep.append(g.index.values); continue
    share = g['Label'].value_counts(normalize=True)
    for lab, p in share.items():
        idx = g.index[g['Label'] == lab].values
        k = min(len(idx), int(round(p * CAP)))
        keep.append(rng.choice(idx, k, replace=False))
E = E.loc[np.sort(np.concatenate(keep))].reset_index(drop=True)
feat_all = [c for c in E.columns if c not in ('day', 'Src IP', 'Timestamp', 'Label', 'Attempted Category', 'family')]
E['h'] = pd.util.hash_pandas_object(E[feat_all], index=False).values           # duplicate key over all 81 non-identifier features
E['y'] = E['family'].map({f: i for i, f in enumerate(FAM)}).astype(int)
E['ts'] = pd.to_datetime(E['Timestamp'], format='mixed')
print(f'\nEnterprise sample: {len(E):,} rows'); print(E.groupby(['family', 'Label']).size().to_string())
dup = E.duplicated('h', keep=False)
g = E.groupby('h')['y'].nunique()
print(f'Exact-duplicate rows (81 features): {dup.mean() * 100:.2f}% | duplicate groups with conflicting families: {int((g > 1).sum())}')

def split_random(df):
    grp = df.groupby('h')['y'].agg(lambda s: s.mode().iloc[0]).reset_index()
    tr, rest = train_test_split(grp, test_size=0.30, stratify=grp['y'], random_state=SEED)
    va, te = train_test_split(rest, test_size=0.50, stratify=rest['y'], random_state=SEED)
    lab = {**{h: 'train' for h in tr.h}, **{h: 'val' for h in va.h}, **{h: 'test' for h in te.h}}
    part = df['h'].map(lab)
    return {k: np.where(part.values == k)[0] for k in ('train', 'val', 'test')}

def split_time(df):
    out = {'train': [], 'val': [], 'test': []}
    for y, g in df.groupby('y'):
        g = g.sort_values(['ts']).index.values; n = len(g); a, b = int(0.70 * n), int(0.85 * n)
        out['train'] += list(g[:a]); out['val'] += list(g[a:b]); out['test'] += list(g[b:])
    out = {k: np.array(sorted(v)) for k, v in out.items()}
    trh = set(df['h'].values[out['train']])
    drop = {}
    for k in ('val', 'test'):
        m = np.array([h not in trh for h in df['h'].values[out[k]]]); drop[k] = int((~m).sum()); out[k] = out[k][m]
    print(f'   time-block split: dropped val/test rows with an exact train twin: {drop}')
    for y, g in df.groupby('y'):
        tr_max = df['ts'].values[np.intersect1d(out['train'], g.index.values)].max()
        te_min = df['ts'].values[np.intersect1d(out['test'], g.index.values)].min()
        assert tr_max <= te_min, f'time order violated for class {FAM[y]} - STOP'
    return out

def finish(Xraw, y, ip, parts, name, domain):
    Xr = Xraw.replace([np.inf, -np.inf], np.nan)
    med = Xr.iloc[parts['train']].median()
    Xf = Xr.fillna(med)
    sc = StandardScaler().fit(Xf.iloc[parts['train']])
    Z = sc.transform(Xf).astype('float32')
    assert np.isfinite(Z).all()
    for k in ('train', 'val', 'test'):
        assert set(np.unique(y[parts[k]])) == set(range(len(FAM))), f'{domain}/{name}: a family is missing from {k} - STOP'
    np.savez_compressed(f'{XSD}/{domain}_{name}.npz', **{f'X_{k}': Z[parts[k]] for k in parts}, **{f'y_{k}': y[parts[k]] for k in parts},
                        **{f'idx_{k}': parts[k] for k in parts}, ip_train=ip[parts['train']].astype(str))
    joblib.dump({'scaler': sc, 'medians': med, 'columns': CIC68, 'families': FAM}, f'{XSD}/{domain}_{name}_prep.pkl')
    tab = pd.DataFrame({k: pd.Series(y[parts[k]]).map(dict(enumerate(FAM))).value_counts() for k in parts}).reindex(FAM)
    print(f'\n{domain} / {name}: train {len(parts["train"]):,} val {len(parts["val"]):,} test {len(parts["test"]):,}'); print(tab.to_string())
    summary[f'{domain}_{name}'] = {'counts': tab.fillna(0).astype(int).to_dict(), 'n': {k: int(len(v)) for k, v in parts.items()}}

for name, fn in (('random', split_random), ('time', split_time)):
    parts = fn(E)
    if name == 'random':
        hs = {k: set(E['h'].values[v]) for k, v in parts.items()}
        assert not (hs['train'] & hs['val']) and not (hs['train'] & hs['test']) and not (hs['val'] & hs['test']), 'duplicate group crosses a split - STOP'
    finish(E[CIC68], E['y'].values, E['Src IP'].values, parts, name, 'ent')
print(f'\nenterprise source IPs in the sample: {E["Src IP"].nunique()} | {(time.time() - t0) / 60:.1f} min')

# ====================== IoT: CIC IoT-DIAD 2024, re-using the leak-free split indices ======================
raw = pd.read_parquet(f'{D}/raw_sampled.parquet')
mapI = {'Benign': 'Benign', 'Brute Force': 'Brute Force', 'DDOS': 'DDoS', 'DOS': 'DoS', 'Recon': 'Recon', 'Web-Based': 'Web'}
yI_all = raw['class_label'].map(lambda c: FAM.index(mapI[c]) if c in mapI else -1).values
XI = raw[CIC68].apply(pd.to_numeric, errors='coerce'); ipI = raw['Src IP'].astype(str).values
for name in ('random', 'time'):
    S = np.load(f'{V2D}/split_{name}.npz')
    parts = {k: S[f'idx_{k}'][yI_all[S[f'idx_{k}']] >= 0] for k in ('train', 'val', 'test')}   # keep the 6 shared families only
    finish(XI, yI_all, ipI, parts, name, 'iot')
print(f'IoT source IPs (shared families, random split train): {len(set(ipI[np.load(f"{XSD}/iot_random.npz")["idx_train"]]))}')

json.dump(summary, open(f'{XSD}/splits_summary.json', 'w'), indent=1)
print(f'\nsaved data/crosssilo/{{iot,ent}}_{{random,time}}.npz + prep files + splits_summary.json | CELL 4 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSS-SILO FL — CELL 5: centralised references + cross-domain transfer (FA-MLP 10 seeds, LightGBM 1 run)
#                         + client-partition preview.  GPU T4, ~1.5-2 h, resumable.
import os, glob, json, time, sys, shutil, importlib, gc
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; CEN = f'{XS}/central'; os.makedirs(CEN, exist_ok=True)
shutil.copy(f'{V2D}/fl_core.py', '/content/fl_core.py'); sys.path.insert(0, '/content')
import fl_core; importlib.reload(fl_core)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip -q install lightgbm'); import lightgbm as lgb
print('GPU:', tf.config.list_physical_devices('GPU'))

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
fl_core.N_CLASSES = NC                                            # harness builds a 5-class head from here on
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
def load(dom, sp):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC                                      # main task: 5 shared families (Web excluded)
        out[f'X_{k}'], out[f'y_{k}'] = S[f'X_{k}'][m], S[f'y_{k}'][m].astype(np.int32)
    out['ip_train'] = S['ip_train'][S['y_train'] < NC]
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(NC)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(NC))]

def train_nn(Xtr, ytr, Xva, yva, seed):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m = fl_core.build_model(Xtr.shape[1], attention=True)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=512, callbacks=[es], verbose=0)
    return m, len(h.history['loss'])

rows = []; t0 = time.time()
for sp in ('random', 'time'):
    DOM = {d: load(d, sp) for d in ('iot', 'ent')}
    sets = {'iot': ('iot',), 'ent': ('ent',), 'pooled': ('iot', 'ent')}
    for tr_name, doms in sets.items():
        Xtr = np.concatenate([DOM[d]['X_train'] for d in doms]); ytr = np.concatenate([DOM[d]['y_train'] for d in doms])
        Xva = np.concatenate([DOM[d]['X_val'] for d in doms]); yva = np.concatenate([DOM[d]['y_val'] for d in doms])
        # ---- LightGBM, one run (seed 42) ----
        fp = f'{CEN}/{sp}_{tr_name}_lgbm_s42.json'
        if not os.path.exists(fp):
            g = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced',
                                   random_state=42, n_jobs=-1, verbose=-1).fit(Xtr, ytr)
            json.dump({te: {'macro_f1': mf1(DOM[te]['y_test'], g.predict(DOM[te]['X_test'])),
                            'per_class': pcf(DOM[te]['y_test'], g.predict(DOM[te]['X_test']))} for te in ('iot', 'ent')}, open(fp, 'w'))
        o = json.load(open(fp))
        for te in ('iot', 'ent'): rows.append({'split': sp, 'train': tr_name, 'model': 'LightGBM', 'seed': 42, 'test': te, **o[te]})
        # ---- FA-MLP, 10 seeds ----
        for s in SEEDS:
            fp = f'{CEN}/{sp}_{tr_name}_famlp_s{s}.json'
            if not os.path.exists(fp):
                m, ep = train_nn(Xtr, ytr, Xva, yva, s)
                res = {'epochs': ep}
                for te in ('iot', 'ent'):
                    p = m.predict(DOM[te]['X_test'], batch_size=8192, verbose=0).argmax(1)
                    res[te] = {'macro_f1': mf1(DOM[te]['y_test'], p), 'per_class': pcf(DOM[te]['y_test'], p)}
                json.dump(res, open(fp, 'w')); del m; gc.collect()
                print(f'  {sp:<6} train={tr_name:<6} FA-MLP s{s:<5} IoT-test {res["iot"]["macro_f1"]:.4f} | Ent-test {res["ent"]["macro_f1"]:.4f}'
                      f' | {ep} ep | {(time.time() - t0) / 60:.1f} min', flush=True)
            o = json.load(open(fp))
            for te in ('iot', 'ent'): rows.append({'split': sp, 'train': tr_name, 'model': 'FA-MLP', 'seed': s, 'test': te, **o[te]})

df = pd.DataFrame(rows); df.drop(columns='per_class').to_csv(f'{XS}/central_summary.csv', index=False)
pd.set_option('display.width', 200)
print('\n===== centralised test macro-F1, mean (SD) over seeds; rows = training data, columns = test domain =====')
t = df.groupby(['split', 'model', 'train', 'test']).macro_f1.agg(['mean', 'std', 'count']).round(4)
print(t.unstack('test').to_string())
print('\n===== H1 transfer gap (in-domain minus cross-domain), FA-MLP, pp =====')
for sp in ('random', 'time'):
    q = df[(df.split == sp) & (df.model == 'FA-MLP')].pivot_table(index='seed', columns=['train', 'test'], values='macro_f1')
    g_ie = 100 * (q[('iot', 'iot')] - q[('ent', 'iot')]); g_ei = 100 * (q[('ent', 'ent')] - q[('iot', 'ent')])
    print(f'  {sp}: on IoT test {g_ie.mean():.2f} pp (min {g_ie.min():.2f}) | on Ent test {g_ei.mean():.2f} pp (min {g_ei.min():.2f})')
pc = df[df.model == 'FA-MLP'].groupby(['split', 'train', 'test']).per_class.apply(lambda s: np.mean(np.vstack(s), 0))
print('\nmean per-class F1 (FA-MLP):'); print(pd.DataFrame(pc.tolist(), index=pc.index, columns=FAM5).round(3).to_string())

# ---- client-partition preview (seed 42): sizes and classes per client, to fix the partition choice ----
print('\n===== partition preview, 10 clients per domain, seed 42 (random split) =====')
for dom in ('iot', 'ent'):
    Dd = load(dom, 'random')
    for kind, a in (('iid', None), ('dirichlet', 0.5), ('dirichlet', 0.1), ('ip', None)):
        parts = [p for p in fl_core.partition(Dd['y_train'], kind, 10, 42, alpha=a, groups=Dd['ip_train'] if kind == 'ip' else None) if len(p)]
        sz = [len(p) for p in parts]; ncls = [int((np.bincount(Dd['y_train'][p], minlength=NC) > 0).sum()) for p in parts]
        print(f'  {dom} {kind}{a if a else "":<5}: {len(parts)} clients | size {min(sz):,}-{max(sz):,} | classes per client {ncls}')
print(f'\nCELL 5 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSS-SILO FL — CELL 6: learning-rate pilot on VALIDATION data only (seed 42, random split, Dirichlet a=0.5)
#                         + design-decision record + measured run time.  GPU T4, ~2 h, resumable.
import os, glob, json, time, sys, importlib, gc, datetime
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; PIL = f'{XS}/pilot'; os.makedirs(PIL, exist_ok=True)

# ---- harness: the paper-1 fl_core.py, unchanged except (i) a 5-class head and (ii) each evaluation keeps its
#      predictions so every domain is scored separately. fl_core.py itself is not modified.
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]

# ---- design decisions, written ONCE, before any federated run (timestamped; never overwritten) ----
DEC = f'{XS}/design_decisions.json'
if not os.path.exists(DEC):
    json.dump({
        'written_utc': datetime.datetime.utcnow().isoformat(timespec='seconds'),
        'written_before_any_federated_run': True,
        'main_task': FAM5, 'sensitivity_task': 'Web added (6 families)',
        'partition_main': 'Dirichlet alpha=0.5, 10 clients per domain, drawn separately in each domain with the run seed; '
                          'cross-silo clients = union of the two within-domain client sets (identical clients)',
        'partition_reference': 'IID, FedAvg only',
        'partition_excluded': 'source-IP clients: enterprise sample has 23 source IPs, 9 of 10 IP clients hold a single class',
        'H1': 'cross-domain transfer loses >= 10 pp macro-F1 in both directions (centralised FA-MLP)',
        'H2': 'kept as stated: cross-silo FedAvg raises each domain test macro-F1 by >= 1 pp over within-domain FedAvg (Holm p<0.05)',
        'H2b_added': 'non-inferiority: cross-silo FedAvg is within 1 pp of within-domain FedAvg on each domain (TOST, alpha 0.05, '
                     '10 paired seeds). Added after the centralised references (pooling: +0.19 pp IoT; enterprise at 0.9993 ceiling), '
                     'before any federated run.',
        'H3': 'under cross-silo training, at least one of FedProx/SCAFFOLD/FedDyn exceeds FedAvg (paired t, Holm)',
        'H4': 'rule ranking differs between within-domain and cross-silo training',
        'lr_selection': 'seed 42, random split, Dirichlet 0.5, VALIDATION only (test not computed): mean of val macro-F1 over rounds '
                        '80/90/100; cross-silo = mean of the two domains; chosen per rule and per setting; FedProx uses the FedAvg lr; '
                        'differences < 0.0005 go to the smaller lr; the same lr is used on the time split',
        'fixed': {'rounds': 100, 'local_epochs': 1, 'batch': 256, 'momentum': 0.9, 'clip_norm': 1.0, 'mu': 0.01, 'dyn_alpha': 0.001},
        'score': 'test macro-F1 averaged over rounds 80/90/100, per domain', 'seeds': SEEDS}, open(DEC, 'w'), indent=1)
    print('design decisions recorded')
else:
    print('design decisions already recorded at', json.load(open(DEC))['written_utc'], 'UTC')

def load(dom, sp):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(NC)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(NC))]

def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    """setting in {'iot','ent','xsilo'}; returns per-domain val (and test) macro-F1 for every evaluated round."""
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, y, Xv, yv = cat('X_train'), cat('y_train'), cat('X_val'), cat('y_val')
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:                                                  # same seed -> same domain clients in every setting
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]])
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]])))
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False

def last3(res, key):
    v = [h[key] for h in res['history'] if h['round'] in (80, 90, 100)]
    return float(np.mean(v)) if len(v) == 3 else np.nan

# ---- pilot ----
DATA = {d: load(d, 'random') for d in ('iot', 'ent')}
GRID = {'fedavg': [0.1, 0.3], 'scaffold': [0.01, 0.03, 0.1], 'feddyn': [0.01, 0.03, 0.1]}
rows = []; t0 = time.time()
for method, lrs in GRID.items():
    for lr in lrs:
        for setting in ('iot', 'ent', 'xsilo'):
            r, cached = run(DATA, setting, 'random', 0.5, method, 42, lr, with_test=False, outdir=PIL)
            doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
            sc = {d: last3(r, f'val_{d}') for d in doms}
            rows.append({'setting': setting, 'method': method, 'lr': lr, 'diverged': bool(r['diverged']),
                         'minutes': r['minutes'], **{f'val_{d}': v for d, v in sc.items()},
                         'val_score': np.nan if r['diverged'] else float(np.mean(list(sc.values())))})
            if not cached:
                print(f'  {setting:<5} {method:<8} lr={lr:<5} ' + ' | '.join(f'val {d} {v:.4f}' for d, v in sc.items())
                      + (f' | DIVERGED' if r['diverged'] else '') + f' | {r["minutes"] or 0:.1f} min | total {(time.time() - t0) / 60:.0f} min', flush=True)

df = pd.DataFrame(rows); df.to_csv(f'{XS}/pilot_summary.csv', index=False)
pd.set_option('display.width', 200)
print('\n===== pilot: validation macro-F1, mean of rounds 80/90/100 (seed 42, random split, Dirichlet 0.5) =====')
print(df.drop(columns='minutes').round(4).to_string(index=False))

chosen = {}
for (setting, method), g in df.groupby(['setting', 'method']):
    g = g[~g.diverged].sort_values('lr')
    if len(g) == 0: chosen.setdefault(setting, {})[method] = None; continue
    best = g.val_score.max(); lr = float(g[g.val_score >= best - 0.0005].lr.iloc[0])   # near-ties go to the smaller lr
    chosen.setdefault(setting, {})[method] = lr
for s in chosen: chosen[s]['fedprox'] = chosen[s]['fedavg']
json.dump(chosen, open(f'{XS}/lr_chosen.json', 'w'), indent=1)
print('\nchosen learning rates:', json.dumps(chosen))

# ---- measured time per run and projected main grid ----
ok = df[~df.diverged & df.minutes.notna()]
tm = {s: float(ok[ok.setting == s].minutes.mean()) for s in ('iot', 'ent', 'xsilo')}
per_cfg = sum(tm.values())
print(f'\nminutes per run: IoT {tm["iot"]:.1f} | Ent {tm["ent"]:.1f} | cross-silo {tm["xsilo"]:.1f}')
print(f'projected main grid (4 rules x 10 seeds x 3 settings): {40 * per_cfg / 60:.1f} h per split '
      f'| random + time {80 * per_cfg / 60:.1f} h | + IID FedAvg reference {10 * per_cfg / 60:.1f} h')
print(f'\nCELL 6 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSS-SILO FL — CELL 7: MAIN GRID (test scores). GPU T4. ~18 h in total, fully resumable:
#   re-run this same cell after every disconnect; finished runs are skipped. Jobs run in priority order.
import os, glob, json, time, sys, importlib, gc
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; GRD = f'{XS}/grid'; os.makedirs(GRD, exist_ok=True)
assert os.path.exists(f'{XS}/design_decisions.json') and os.path.exists(f'{XS}/lr_chosen.json')
LR = json.load(open(f'{XS}/lr_chosen.json'))

# ---- same harness patch as Cell 6 (5-class head; predictions kept per evaluation) ----
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
def load(dom, sp):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(NC)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(NC))]

def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, y, Xv, yv = cat('X_train'), cat('y_train'), cat('X_val'), cat('y_val')
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]])
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]])))
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False
def last3(res, key):
    v = [h[key] for h in res['history'] if h['round'] in (80, 90, 100)]
    return float(np.mean(v)) if len(v) == 3 else np.nan

# ---- job list, in priority order (role: main = pre-specified lr; lrswap = FedAvg lr robustness check) ----
SETS = ('iot', 'ent', 'xsilo'); RULES = ('fedavg', 'feddyn', 'scaffold', 'fedprox')
JOBS = []
add = lambda sp, a, m, st, lr, role: JOBS.extend((sp, a, m, st, s, lr, role) for s in SEEDS)
for st in SETS: add('random', 0.5, 'fedavg', st, LR[st]['fedavg'], 'main')                       # 1. H2/H2b core
sw = {'iot': LR['xsilo']['fedavg'], 'ent': LR['xsilo']['fedavg'], 'xsilo': LR['iot']['fedavg']}
for st in SETS:                                                                                   # 2. lr robustness
    if sw[st] != LR[st]['fedavg']: add('random', 0.5, 'fedavg', st, sw[st], 'lrswap')
for m in RULES[1:]:                                                                               # 3. H3/H4, random
    for st in SETS: add('random', 0.5, m, st, LR[st][m], 'main')
for m in RULES:                                                                                   # 4. time split
    for st in SETS: add('time', 0.5, m, st, LR[st][m], 'main')
for st in SETS: add('random', None, 'fedavg', st, LR[st]['fedavg'], 'main')                      # 5. IID reference
EST = {'iot': 2.7, 'ent': 2.7, 'xsilo': 5.3}

DATA = {sp: {d: load(d, sp) for d in ('iot', 'ent')} for sp in ('random', 'time')}
kind_of = lambda a: 'iid' if a is None else f'dirichlet{a}'
done_fp = lambda j: f'{GRD}/{j[0]}_{kind_of(j[1])}_{j[3]}_{j[2]}_lr{j[5]}_s{j[4]}.json'
todo = [j for j in JOBS if not os.path.exists(done_fp(j))]
print(f'{len(JOBS)} runs in grid | {len(JOBS) - len(todo)} already done | ~{sum(EST[j[3]] for j in todo) / 60:.1f} h left')

t0 = time.time()
for i, (sp, a, m, st, s, lr, role) in enumerate(todo):
    r, cached = run(DATA[sp], st, sp, a, m, s, lr, with_test=True, outdir=GRD)
    doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    left = sum(EST[j[3]] for j in todo[i + 1:]) / 60
    print(f'  [{i + 1}/{len(todo)}] {sp:<6} {kind_of(a):<12} {st:<5} {m:<8} lr={lr:<5} s{s:<5} '
          + ' | '.join(f'test {d} {last3(r, "test_" + d):.4f}' for d in doms)
          + (' | DIVERGED' if r['diverged'] else '') + f' | {r["minutes"] or 0:.1f} min | ~{left:.1f} h left', flush=True)

# ---- summary of everything finished so far (works on a partial grid) ----
rows = []
for j in JOBS:
    if not os.path.exists(done_fp(j)): continue
    sp, a, m, st, s, lr, role = j; r = json.load(open(done_fp(j)))
    for d in (('iot', 'ent') if st == 'xsilo' else (st,)):
        rows.append({'split': sp, 'part': kind_of(a), 'method': m, 'role': role, 'setting': st, 'lr': lr, 'seed': s,
                     'domain': d, 'test': last3(r, f'test_{d}'), 'diverged': bool(r['diverged']), 'minutes': r['minutes']})
df = pd.DataFrame(rows); df.to_csv(f'{XS}/grid_summary.csv', index=False)
pd.set_option('display.width', 220)
df['where'] = np.where(df.setting == 'xsilo', 'xsilo', 'within')
t = df.groupby(['split', 'part', 'method', 'role', 'domain', 'where']).test.agg(['mean', 'std', 'count']).round(4)
print('\n===== test macro-F1 (rounds 80/90/100), mean / SD / n seeds =====')
print(t.unstack('where').to_string())
print('\n===== paired difference, cross-silo minus within-domain (pp), main lr =====')
mm = df[df.role == 'main']
for (sp, part, m, d), g in mm.groupby(['split', 'part', 'method', 'domain']):
    p = g.pivot_table(index='seed', columns='where', values='test').dropna()
    if {'xsilo', 'within'} <= set(p.columns) and len(p):
        dd = 100 * (p.xsilo - p.within)
        print(f'  {sp:<6} {part:<12} {m:<8} {d}: {dd.mean():+.2f} pp (SD {dd.std():.2f}, min {dd.min():+.2f}, max {dd.max():+.2f}, n {len(dd)})')
print(f'\ndiverged runs: {int(df.drop_duplicates(["split","part","method","setting","lr","seed"]).diverged.sum())}'
      f' | CELL 7 session time {(time.time() - t0) / 60:.1f} min | runs finished {df.drop_duplicates(["split","part","method","setting","lr","seed"]).shape[0]}/{len(JOBS)}')

In [ ]:
# CROSS-SILO FL — CELL 8: export every result file + split/partition counts into one zip (CPU is fine, ~2-4 min)
import os, glob, json, zipfile, io
import numpy as np, pandas as pd
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'
import sys, shutil; shutil.copy(f'{V2D}/fl_core.py', '/content/fl_core_ref.py'); sys.path.insert(0, '/content')
import fl_core_ref as fc

FAM6 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon', 'Web']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]

# ---- 1. class counts per domain / split / part (all 6 families) ----
cc = []
for dom in ('iot', 'ent'):
    for sp in ('random', 'time'):
        S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True)
        for k in ('train', 'val', 'test'):
            b = np.bincount(S[f'y_{k}'], minlength=6)
            cc += [{'domain': dom, 'split': sp, 'part': k, 'family': FAM6[c], 'n': int(b[c])} for c in range(6)]
cc = pd.DataFrame(cc)

# ---- 2. client composition (5-family task) for every seed, partition and domain, both splits ----
pc = []
for dom in ('iot', 'ent'):
    for sp in ('random', 'time'):
        S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); y = S['y_train'][S['y_train'] < NC]
        for kind, a in (('iid', None), ('dirichlet', 0.5)):
            for s in SEEDS:
                for k, p in enumerate(fc.partition(y, kind, 10, s, alpha=a)):
                    b = np.bincount(y[p], minlength=NC)
                    pc.append({'domain': dom, 'split': sp, 'partition': f'{kind}{a if a else ""}', 'seed': s, 'client': k,
                               'size': int(len(p)), **{FAM6[c]: int(b[c]) for c in range(NC)}})
pc = pd.DataFrame(pc)

# ---- 3. zip: summaries, decisions, every run JSON (central, pilot, grid), the two tables above ----
zp = '/content/export_crosssilo_main.zip'; n = 0
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in ('design_decisions.json', 'lr_chosen.json', 'pilot_summary.csv', 'central_summary.csv', 'grid_summary.csv'):
        z.write(f'{XS}/{f}', f); n += 1
    z.write(f'{XSD}/splits_summary.json', 'splits_summary.json'); n += 1
    for sub in ('central', 'pilot', 'grid'):
        for f in sorted(glob.glob(f'{XS}/{sub}/*.json')):
            z.write(f, f'{sub}/{os.path.basename(f)}'); n += 1
    z.writestr('class_counts.csv', cc.to_csv(index=False)); z.writestr('client_composition.csv', pc.to_csv(index=False)); n += 2
print({sub: len(glob.glob(f'{XS}/{sub}/*.json')) for sub in ('central', 'pilot', 'grid')})
print(f'{n} files | zip {os.path.getsize(zp) / 1e6:.1f} MB')
print(cc.pivot_table(index=['domain', 'split', 'family'], columns='part', values='n').astype(int).to_string())
shutil.copy(zp, f'{XS}/export_crosssilo_main.zip')
files.download(zp)

In [ ]:
# CROSS-SILO FL — CELL 9: the two pre-specified sensitivity checks (FedAvg, random split, 10 seeds, 3 settings)
#   (a) 6 families (Web added), Dirichlet 0.5   (b) 5 families, Dirichlet 0.1.  GPU T4, ~3.6 h, resumable.
import os, glob, json, time, sys, importlib, gc
import numpy as np, pandas as pd
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; GRD = f'{XS}/sens'; os.makedirs(GRD, exist_ok=True)
assert os.path.exists(f'{XS}/design_decisions.json') and os.path.exists(f'{XS}/lr_chosen.json')
LR = json.load(open(f'{XS}/lr_chosen.json'))

# ---- same harness patch as Cell 6 (5-class head; predictions kept per evaluation) ----
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
def load(dom, sp, nc):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < nc
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(fl_xs.N_CLASSES)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(fl_xs.N_CLASSES))]

def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'c{fl_xs.N_CLASSES}_{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, y, Xv, yv = cat('X_train'), cat('y_train'), cat('X_val'), cat('y_val')
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]])
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]])))
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False
def last3(res, key):
    v = [h[key] for h in res['history'] if h['round'] in (80, 90, 100)]
    return float(np.mean(v)) if len(v) == 3 else np.nan

# ---- jobs: (n classes, Dirichlet alpha); FedAvg at the pre-specified learning rate of each setting ----
SETS = ('iot', 'ent', 'xsilo'); EST = {'iot': 2.7, 'ent': 2.7, 'xsilo': 5.3}
CHECKS = [(6, 0.5), (5, 0.1)]
JOBS = [(nc, a, st, s) for nc, a in CHECKS for st in SETS for s in SEEDS]
fp_of = lambda nc, a, st, s: f'{GRD}/c{nc}_random_dirichlet{a}_{st}_fedavg_lr{LR[st]["fedavg"]}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fp_of(*j))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{sum(EST[j[2]] for j in todo) / 60:.1f} h left')
t0 = time.time(); DATA = {}
for i, (nc, a, st, s) in enumerate(todo):
    if nc not in DATA: DATA = {nc: {d: load(d, 'random', nc) for d in ('iot', 'ent')}}
    fl_xs.N_CLASSES = nc                                   # head size and F1 labels follow the task
    r, _ = run(DATA[nc], st, 'random', a, 'fedavg', s, LR[st]['fedavg'], with_test=True, outdir=GRD)
    doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    print(f'  [{i + 1}/{len(todo)}] {nc} classes  dirichlet{a:<4} {st:<5} s{s:<5} '
          + ' | '.join(f'test {d} {last3(r, "test_" + d):.4f}' for d in doms)
          + (' | DIVERGED' if r['diverged'] else '') + f' | {r["minutes"] or 0:.1f} min'
          + f' | ~{sum(EST[j[2]] for j in todo[i + 1:]) / 60:.1f} h left', flush=True)

# ---- summary: paired cross-silo minus within-domain, per domain; reference = main grid (5 classes, Dirichlet 0.5) ----
from scipy import stats
def score(path, d): return last3(json.load(open(path)), f'test_{d}')
ref = lambda st, s: f'{XS}/grid/random_dirichlet0.5_{st}_fedavg_lr{LR[st]["fedavg"]}_s{s}.json'
print('\n===== FedAvg, random split: cross-silo minus within-domain (pp), 10 paired seeds =====')
rows = []
for name, f in [('main: 5 classes, Dirichlet 0.5', ref)] + [(f'{nc} classes, Dirichlet {a}', (lambda nc, a: lambda st, s: fp_of(nc, a, st, s))(nc, a)) for nc, a in CHECKS]:
    for d in ('iot', 'ent'):
        if not all(os.path.exists(f(st, s)) for st in (d, 'xsilo') for s in SEEDS): print(f'  {name} {d}: incomplete'); continue
        w = np.array([score(f(d, s), d) for s in SEEDS]); x = np.array([score(f('xsilo', s), d) for s in SEEDS]); dd = 100 * (x - w)
        se = dd.std(ddof=1) / np.sqrt(10)
        rows.append({'check': name, 'domain': d, 'within': w.mean(), 'xsilo': x.mean(), 'diff_pp': dd.mean(), 'sd': dd.std(ddof=1),
                     'neg': int((dd < 0).sum()), 'p': stats.ttest_rel(x, w).pvalue,
                     'p_tost1pp': max(stats.t.sf((dd.mean() + 1) / se, 9), stats.t.cdf((dd.mean() - 1) / se, 9))})
pd.set_option('display.width', 220); print(pd.DataFrame(rows).round(4).to_string(index=False))
if all(os.path.exists(fp_of(6, 0.5, st, s)) for st in SETS for s in SEEDS):
    print('\nWeb F1 (6-class task), mean over seeds:',
          {d: {'within': round(float(np.mean([last3(json.load(open(fp_of(6, 0.5, d, s))), f'test_pc_{d}')[5] for s in SEEDS])), 4),
               'xsilo': round(float(np.mean([last3(json.load(open(fp_of(6, 0.5, 'xsilo', s))), f'test_pc_{d}')[5] for s in SEEDS])), 4)} for d in ('iot', 'ent')})
print(f'\nCELL 9 session time {(time.time() - t0) / 60:.1f} min | finished {sum(os.path.exists(fp_of(*j)) for j in JOBS)}/{len(JOBS)}')

In [ ]:
# CROSS-SILO FL — CELL 10: export the sensitivity runs (CPU is fine, ~1 min). Last Colab cell for this paper.
import os, glob, zipfile, shutil
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; XS = f'{DRIVE_PATH}/results/crosssilo'
fs = sorted(glob.glob(f'{XS}/sens/*.json')); assert len(fs) == 60, len(fs)
zp = '/content/export_crosssilo_sens.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in fs: z.write(f, f'sens/{os.path.basename(f)}')
print(f'{len(fs)} files | zip {os.path.getsize(zp) / 1e6:.2f} MB')
shutil.copy(zp, f'{XS}/export_crosssilo_sens.zip'); files.download(zp)

In [ ]:
# CROSS-SILO FL — CELL 11: label-separation diagnostic (EXPLORATORY, added after the main results).
#   Cross-silo training where each domain keeps its OWN output units (10 = 5 families x 2 domains);
#   at test time a predicted unit is mapped back to its family, so no domain information is used for scoring.
#   If the IoT cost disappears, the cost comes from forcing two domains' traffic into one shared label.
#   4 rules x 10 seeds, random split, Dirichlet 0.5, cross-silo learning rates. GPU T4, ~3.6 h, resumable.
import os, glob, json, time, sys, importlib, gc
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; GRD = f'{XS}/sepheads'; os.makedirs(GRD, exist_ok=True)
assert os.path.exists(f'{XS}/design_decisions.json') and os.path.exists(f'{XS}/lr_chosen.json')
LR = json.load(open(f'{XS}/lr_chosen.json'))

# ---- same harness patch as Cell 6 (5-class head; predictions kept per evaluation) ----
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
def load(dom, sp, nc):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < nc
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(NC)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(NC))]
SEP = True

def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'sep_{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, Xv = cat('X_train'), cat('X_val'); yv = cat('y_val')
    y = np.concatenate([DATA[d]['y_train'] + (5 if (SEP and d == 'ent') else 0) for d in doms]).astype(np.int32)
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]   # same clients as main grid
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]] % 5)
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]] % 5)
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]] % 5)
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]] % 5)))
                rec[f'other_dom_units_{d}'] = float(np.mean((h['pt'][ct[i]:ct[i + 1]] >= 5) != (i == 1)))  # share predicted via the other domain's units
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False
def last3(res, key):
    v = [h[key] for h in res['history'] if h['round'] in (80, 90, 100)]
    return float(np.mean(v)) if len(v) == 3 else np.nan

# ---- jobs ----
fl_xs.N_CLASSES = 10                                       # 5 families x 2 domains, training only
RULES = ('fedavg', 'feddyn', 'scaffold', 'fedprox')
JOBS = [(m, s) for m in RULES for s in SEEDS]
fp_of = lambda m, s: f'{GRD}/sep_random_dirichlet0.5_xsilo_{m}_lr{LR["xsilo"][m]}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fp_of(*j))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{len(todo) * 5.4 / 60:.1f} h left')
DATA = {d: load(d, 'random', NC) for d in ('iot', 'ent')}; t0 = time.time()
for i, (m, s) in enumerate(todo):
    r, _ = run(DATA, 'xsilo', 'random', 0.5, m, s, LR['xsilo'][m], with_test=True, outdir=GRD)
    main = json.load(open(f'{XS}/grid/random_dirichlet0.5_xsilo_{m}_lr{LR["xsilo"][m]}_s{s}.json'))
    assert r['client_sizes'] == main['client_sizes'], 'clients differ from main grid'
    print(f'  [{i + 1}/{len(todo)}] sep-heads {m:<8} s{s:<5} test iot {last3(r, "test_iot"):.4f} | test ent {last3(r, "test_ent"):.4f}'
          + (' | DIVERGED' if r['diverged'] else '') + f' | {r["minutes"] or 0:.1f} min | ~{(len(todo) - i - 1) * 5.4 / 60:.1f} h left', flush=True)

# ---- summary: separate heads vs shared heads (main cross-silo) vs within-domain, 10 paired seeds ----
from scipy import stats
g = lambda st, m, s: f'{XS}/grid/random_dirichlet0.5_{st}_{m}_lr{LR[st][m]}_s{s}.json'
L = lambda p, k: (lambda r: np.mean([h[k] for h in r['history'] if h['round'] in (80, 90, 100)], 0))(json.load(open(p)))
print('\n===== random split, Dirichlet 0.5: test macro-F1, mean over 10 seeds; paired differences in pp =====')
for m in RULES:
    if not all(os.path.exists(fp_of(m, s)) for s in SEEDS): print(f'  {m}: incomplete'); continue
    for d in ('iot', 'ent'):
        w = np.array([L(g(d, m, s), f'test_{d}') for s in SEEDS]); x = np.array([L(g('xsilo', m, s), f'test_{d}') for s in SEEDS])
        sp_ = np.array([L(fp_of(m, s), f'test_{d}') for s in SEEDS]); o = np.mean([L(fp_of(m, s), f'other_dom_units_{d}') for s in SEEDS])
        a, b = 100 * (sp_ - w), 100 * (sp_ - x)
        print(f'  {m:<8} {d}: within {w.mean():.4f} | shared {x.mean():.4f} | separate {sp_.mean():.4f} || '
              f'sep-within {a.mean():+.2f} pp (neg {int((a < 0).sum())}/10, p {stats.ttest_rel(sp_, w).pvalue:.4f}) | '
              f'sep-shared {b.mean():+.2f} pp (pos {int((b > 0).sum())}/10, p {stats.ttest_rel(sp_, x).pvalue:.4f}) | via other-domain units {100 * o:.2f}%')
print(f'\nCELL 11 session time {(time.time() - t0) / 60:.1f} min | finished {sum(os.path.exists(fp_of(*j)) for j in JOBS)}/{len(JOBS)}')

In [ ]:
# CROSS-SILO FL — CELL 12: reviewer robustness checks (run AFTER Cell 11 finishes). GPU T4, ~3.7 h, resumable.
#   (A) GLOBAL normalisation: one scaler fitted on the union of both silos' training data (what federated mean/variance
#       aggregation would give) instead of per-silo scalers. Does the IoT cost survive?  FedAvg + FedDyn, 10 seeds.
#   (B) SILO-SIZE ratio: enterprise silo cut to 25% / IoT silo cut to 25% (stratified per family, fixed subsample).
#       Does the IoT cost grow with the other silo's share of the data?  FedAvg, 10 seeds.
#   (C) DOMAIN SEPARABILITY (CPU, minutes): per family, can a classifier tell IoT flows from enterprise flows?
#       (LightGBM AUC on raw features) + mean two-sample KS statistic over the 68 features.
#   (D) CONFUSION MATRICES + reproducibility: main-grid FedAvg runs (IoT / enterprise / cross-silo) re-run for seeds
#       42, 123, 456 with the final-round confusion matrix saved; scores must match the main grid.
import os, glob, json, time, sys, importlib, gc
import numpy as np, pandas as pd
from sklearn.metrics import f1_score, confusion_matrix
from google.colab import drive
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; RB = f'{XS}/robust'; os.makedirs(RB, exist_ok=True)
assert os.path.exists(f'{XS}/design_decisions.json') and os.path.exists(f'{XS}/lr_chosen.json')
LR = json.load(open(f'{XS}/lr_chosen.json'))

# ---- same harness patch as Cell 6 (5-class head; predictions kept per evaluation) ----
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)

FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
def load(dom, sp):
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=np.arange(NC)))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=np.arange(NC))]

def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, y, Xv, yv = cat('X_train'), cat('y_train'), cat('X_val'), cat('y_val')
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]])
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]])))
                if h['round'] == 100: rec[f'cm_{d}'] = confusion_matrix(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]], labels=np.arange(NC)).tolist()
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False
def last3(res, key):
    v = [h[key] for h in res['history'] if h['round'] in (80, 90, 100)]
    return float(np.mean(v)) if len(v) == 3 else np.nan


import joblib
from sklearn.preprocessing import StandardScaler
def find_scaler(obj):
    if hasattr(obj, 'inverse_transform'): return obj
    if isinstance(obj, dict):
        for v in obj.values():
            s = find_scaler(v)
            if s is not None: return s
    return None
BASE = {d: load(d, 'random') for d in ('iot', 'ent')}
SC = {}
for d in ('iot', 'ent'):
    pk = joblib.load(f'{XSD}/{d}_random_prep.pkl')
    print(d, 'prep file keys:', list(pk.keys()) if isinstance(pk, dict) else type(pk))
    SC[d] = find_scaler(pk); assert SC[d] is not None and SC[d].n_features_in_ == BASE[d]['X_train'].shape[1], 'scaler not found'
RAW = {d: {k: SC[d].inverse_transform(BASE[d][k].astype(np.float64)) if k.startswith('X_') else BASE[d][k] for k in BASE[d]} for d in BASE}
# sanity: re-scaling the raw values with the silo's own scaler gives back the stored arrays
for d in BASE: assert np.allclose(SC[d].transform(RAW[d]['X_test'][:2000]), BASE[d]['X_test'][:2000], rtol=1e-4, atol=1e-2), 'inverse check failed'

def subsample(Dd, frac, seed=2026):
    out = {}; rng = np.random.default_rng(seed); y = Dd['y_train']
    keep = np.sort(np.concatenate([rng.choice(np.where(y == c)[0], max(1, int(round(frac * (y == c).sum()))), replace=False)
                                   for c in np.unique(y)]))
    for k in Dd: out[k] = Dd[k][keep] if k in ('X_train', 'y_train') else Dd[k]   # val/test untouched
    return out

g = lambda st, m, s: json.load(open(f'{XS}/grid/random_dirichlet0.5_{st}_{m}_lr{LR[st][m]}_s{s}.json'))
JOBS = []
# (A) global scaler
gs = StandardScaler().fit(np.concatenate([RAW['iot']['X_train'], RAW['ent']['X_train']]))
GLOB = {d: {k: gs.transform(RAW[d][k]).astype(np.float32) if k.startswith('X_') else RAW[d][k] for k in RAW[d]} for d in RAW}
for m in ('fedavg', 'feddyn'):
    JOBS += [('A_globalscaler', GLOB, 'xsilo', m, s) for s in SEEDS]
# (B) silo-size ratio
E25 = {'iot': BASE['iot'], 'ent': subsample(BASE['ent'], 0.25)}
I25 = {'iot': subsample(BASE['iot'], 0.25), 'ent': BASE['ent']}
for s in SEEDS: JOBS += [('B_ent25', E25, 'xsilo', 'fedavg', s), ('B_ent25', E25, 'ent', 'fedavg', s),
                         ('B_iot25', I25, 'xsilo', 'fedavg', s), ('B_iot25', I25, 'iot', 'fedavg', s)]
for s in (42, 123, 456): JOBS += [('D_confusion', BASE, st, 'fedavg', s) for st in ('iot', 'ent', 'xsilo')]
print({n: {d: len(Dv[d]['y_train']) for d in Dv} for n, Dv in (('E25', E25), ('I25', I25))})
EST = {'iot': 2.7, 'ent': 2.7, 'xsilo': 5.3}
fp_of = lambda v, st, m, s: f'{RB}/{v}/random_dirichlet0.5_{st}_{m}_lr{LR[st][m]}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fp_of(j[0], j[2], j[3], j[4]))]
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{sum(EST[j[2]] for j in todo) / 60 * 0.8:.1f} h left')
t0 = time.time()
for i, (v, Dv, st, m, s) in enumerate(todo):
    os.makedirs(f'{RB}/{v}', exist_ok=True)
    r, _ = run(Dv, st, 'random', 0.5, m, s, LR[st][m], with_test=True, outdir=f'{RB}/{v}')
    doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    print(f'  [{i + 1}/{len(todo)}] {v:<15} {st:<5} {m:<7} s{s:<5} ' + ' | '.join(f'test {d} {last3(r, "test_" + d):.4f}' for d in doms)
          + (' | DIVERGED' if r['diverged'] else '') + f' | {r["minutes"] or 0:.1f} min | {(time.time() - t0) / 60:.0f} min total', flush=True)

# ---- summaries ----
from scipy import stats
def paired(a, b, label):
    d = 100 * (np.array(b) - np.array(a)); p = stats.ttest_rel(b, a).pvalue
    return f'{label}: {np.mean(a):.4f} -> {np.mean(b):.4f} = {d.mean():+.2f} pp (SD {d.std(ddof=1):.2f}, neg {int((d < 0).sum())}/10, p {p:.4f})'
L3 = lambda r, k: last3(r, k)
print('\n===== (A) global scaler, cross-silo vs within-domain (main grid, per-silo scaler) =====')
for m in ('fedavg', 'feddyn'):
    if not all(os.path.exists(fp_of('A_globalscaler', 'xsilo', m, s)) for s in SEEDS): print('  incomplete', m); continue
    for d in ('iot', 'ent'):
        w = [L3(g(d, m, s), f'test_{d}') for s in SEEDS]; xs = [L3(g('xsilo', m, s), f'test_{d}') for s in SEEDS]
        xg = [L3(json.load(open(fp_of('A_globalscaler', 'xsilo', m, s))), f'test_{d}') for s in SEEDS]
        print('  ' + paired(w, xg, f'{m} {d} within -> cross-silo GLOBAL scaler') + '\n  ' + paired(xs, xg, f'{m} {d} cross-silo per-silo -> GLOBAL scaler'))
print('\n===== (B) silo-size ratio (FedAvg) =====')
if all(os.path.exists(fp_of('B_ent25', 'xsilo', 'fedavg', s)) for s in SEEDS):
    w = [L3(g('iot', 'fedavg', s), 'test_iot') for s in SEEDS]
    print('  ' + paired(w, [L3(g('xsilo', 'fedavg', s), 'test_iot') for s in SEEDS], 'IoT, enterprise 100% (main)'))
    print('  ' + paired(w, [L3(json.load(open(fp_of('B_ent25', 'xsilo', 'fedavg', s))), 'test_iot') for s in SEEDS], 'IoT, enterprise 25%'))
    print('  ' + paired([L3(json.load(open(fp_of('B_ent25', 'ent', 'fedavg', s))), 'test_ent') for s in SEEDS],
                        [L3(json.load(open(fp_of('B_ent25', 'xsilo', 'fedavg', s))), 'test_ent') for s in SEEDS], 'Enterprise-25% silo'))
if all(os.path.exists(fp_of('B_iot25', st, 'fedavg', s)) for st in ('iot', 'xsilo') for s in SEEDS):
    print('  ' + paired([L3(json.load(open(fp_of('B_iot25', 'iot', 'fedavg', s))), 'test_iot') for s in SEEDS],
                        [L3(json.load(open(fp_of('B_iot25', 'xsilo', 'fedavg', s))), 'test_iot') for s in SEEDS], 'IoT-25% silo, enterprise 100%'))
    print('  ' + paired([L3(g('ent', 'fedavg', s), 'test_ent') for s in SEEDS],
                        [L3(json.load(open(fp_of('B_iot25', 'xsilo', 'fedavg', s))), 'test_ent') for s in SEEDS], 'Enterprise, IoT 25%'))

print('\n===== (D) reproducibility vs main grid + IoT confusion (rows = true, row-normalised, seeds 42/123/456 summed) =====')
if all(os.path.exists(fp_of('D_confusion', st, 'fedavg', s)) for st in ('iot', 'ent', 'xsilo') for s in (42, 123, 456)):
    for st in ('iot', 'ent', 'xsilo'):
        for d in (('iot', 'ent') if st == 'xsilo' else (st,)):
            dd = [abs(L3(json.load(open(fp_of('D_confusion', st, 'fedavg', s))), f'test_{d}') - L3(g(st, 'fedavg', s), f'test_{d}')) for s in (42, 123, 456)]
            print(f'  {st:<5} {d}: max |re-run - main grid| = {max(dd):.5f}')
    for st in ('iot', 'xsilo'):
        cm = sum(np.array(json.load(open(fp_of('D_confusion', st, 'fedavg', s)))['history'][-1]['cm_iot']) for s in (42, 123, 456))
        print(f'  IoT test, {"within-domain" if st == "iot" else "cross-silo"}:')
        print(pd.DataFrame(cm / cm.sum(1, keepdims=True), index=FAM5, columns=FAM5).round(3).to_string())

# ---- (C) domain separability on RAW features (training rows only) ----
try:
    import lightgbm as lgb
except ImportError:
    os.system('pip -q install lightgbm'); import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
fpC = f'{RB}/C_domain_separability.json'
if not os.path.exists(fpC):
    rng = np.random.default_rng(0); outC = {}
    for c in list(range(NC)) + ['all']:
        idx = {d: (np.arange(len(RAW[d]['y_train'])) if c == 'all' else np.where(RAW[d]['y_train'] == c)[0]) for d in RAW}
        idx = {d: rng.choice(v, min(len(v), 5000), replace=False) for d, v in idx.items()}
        Xd = np.concatenate([RAW['iot']['X_train'][idx['iot']], RAW['ent']['X_train'][idx['ent']]])
        yd = np.r_[np.zeros(len(idx['iot'])), np.ones(len(idx['ent']))]
        perm = rng.permutation(len(yd)); cut = int(0.7 * len(yd)); tr, te = perm[:cut], perm[cut:]
        clf = lgb.LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=0, verbose=-1).fit(Xd[tr], yd[tr])
        auc = float(roc_auc_score(yd[te], clf.predict_proba(Xd[te])[:, 1]))
        ks = [ks_2samp(RAW['iot']['X_train'][idx['iot'], j], RAW['ent']['X_train'][idx['ent'], j]).statistic for j in range(Xd.shape[1])]
        outC[FAM5[c] if c != 'all' else 'all'] = {'n_iot': int(len(idx['iot'])), 'n_ent': int(len(idx['ent'])), 'domain_auc': auc,
                                                  'ks_mean': float(np.mean(ks)), 'ks_median': float(np.median(ks)),
                                                  'n_features_ks_gt_0.5': int(np.sum(np.array(ks) > 0.5))}
    json.dump(outC, open(fpC, 'w'), indent=1)
print('\n===== (C) domain separability per family (IoT vs enterprise, raw features) =====')
print(pd.DataFrame(json.load(open(fpC))).T.round(4).to_string())
print(f'\nCELL 12 session time {(time.time() - t0) / 60:.1f} min | finished {sum(os.path.exists(fp_of(j[0], j[2], j[3], j[4])) for j in JOBS)}/{len(JOBS)}')

In [ ]:
# CROSS-SILO FL — CELL 13: standard-comparison metrics + export (run AFTER Cell 12). GPU T4, ~4.5 h, fully resumable.
#   Step 0  export the Cell 11 (separate heads) and Cell 12 (robustness) result files -> export_xs_robust.zip (download)
#   Step 1  centralised FA-MLP, 10 seeds, random split, trained on IoT / enterprise / pooled: confusion matrices on BOTH
#           test domains; macro-F1 checked against the stored Cell 5 value of every run (same recipe, same seeds)
#   Step 2  seven classical baselines (fixed, untuned settings; seed 42), both splits, trained on IoT / enterprise / pooled,
#           scored on both test domains (confusion matrices); LightGBM checked against Cell 5
#   Step 3  federated FedAvg + FedDyn, random split, Dirichlet 0.5, within-domain and cross-silo, 10 seeds: round-100
#           confusion matrices (the 3 FedAvg seeds of Cell 12-D are reused); every run checked against the main grid
#   Step 4  export everything new -> export_xs_std.zip (download)
#   Accuracy, precision, recall, weighted F1 and MCC are computed later from the saved confusion matrices.
import os, glob, json, time, sys, importlib, gc, zipfile
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, confusion_matrix
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; RB = f'{XS}/robust'; ST = f'{XS}/stdmetrics'; os.makedirs(ST, exist_ok=True)
LR = json.load(open(f'{XS}/lr_chosen.json'))

def export(zname, patterns):
    zp = f'{XS}/{zname}'; n = 0
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
        for pat in patterns:
            for f in sorted(glob.glob(f'{XS}/{pat}', recursive=True)):
                if os.path.isfile(f): z.write(f, os.path.relpath(f, XS)); n += 1
    print(f'exported {n} files -> results/crosssilo/{zname} ({os.path.getsize(zp) / 1e6:.1f} MB)'); files.download(zp)

# ---- Step 0 ----
export('export_xs_robust.zip', ['sepheads/*.json', 'robust/**/*.json'])

# ---- harness patch (identical to Cells 6-12) ----
src = open(f'{V2D}/fl_core.py').read()
A = "            rec = {'round': r, 'val_macro_f1'"
B = "rec['n_pred_classes'] = int(len(np.unique(pt)))"
assert src.count(A) == 1 and src.count(B) == 1 and src.count('N_CLASSES = 8') == 1
src = (src.replace(A, "            rec = {'pv': pv.astype(np.uint8), 'round': r, 'val_macro_f1'")
          .replace(B, B + "; rec['pt'] = pt.astype(np.uint8)").replace('N_CLASSES = 8', 'N_CLASSES = 5'))
open('/content/fl_xs.py', 'w').write(src); sys.path.insert(0, '/content')
import fl_xs; importlib.reload(fl_xs)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xs.N_CLASSES)
FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5; LAB = np.arange(NC)
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=LAB))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=LAB)]
cmx = lambda y, p: confusion_matrix(y, p, labels=LAB).tolist()

def load_c5(dom, sp):              # loader of Cell 5 (centralised references), unchanged
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'], out[f'y_{k}'] = S[f'X_{k}'][m], S[f'y_{k}'][m].astype(np.int32)
    return out
def load_fl(dom, sp):              # loader of Cells 7-12 (federated runs), unchanged
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
SETS = {'iot': ('iot',), 'ent': ('ent',), 'pooled': ('iot', 'ent')}
t0 = time.time()

# ---- Step 1: centralised FA-MLP (recipe of Cell 5) ----
def train_nn(Xtr, ytr, Xva, yva, seed):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    m = fl_xs.build_model(Xtr.shape[1], attention=True)
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy')
    es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    h = m.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=60, batch_size=512, callbacks=[es], verbose=0)
    return m, len(h.history['loss'])
DOM = {d: load_c5(d, 'random') for d in ('iot', 'ent')}
dmax = []
for tr, doms in SETS.items():
    Xtr = np.concatenate([DOM[d]['X_train'] for d in doms]); ytr = np.concatenate([DOM[d]['y_train'] for d in doms])
    Xva = np.concatenate([DOM[d]['X_val'] for d in doms]); yva = np.concatenate([DOM[d]['y_val'] for d in doms])
    for s in SEEDS:
        fp = f'{ST}/cen_random_{tr}_famlp_s{s}.json'
        if not os.path.exists(fp):
            tt = time.time(); m, ep = train_nn(Xtr, ytr, Xva, yva, s); fit_s = time.time() - tt; res = {'epochs': ep, 'fit_s': fit_s, 'params': int(m.count_params())}
            for te in ('iot', 'ent'):
                p = m.predict(DOM[te]['X_test'], batch_size=8192, verbose=0).argmax(1)
                res[te] = {'macro_f1': mf1(DOM[te]['y_test'], p), 'per_class': pcf(DOM[te]['y_test'], p), 'cm': cmx(DOM[te]['y_test'], p)}
            json.dump(res, open(fp, 'w')); del m; gc.collect()
        r = json.load(open(fp)); old = json.load(open(f'{XS}/central/random_{tr}_famlp_s{s}.json'))
        d_ = max(abs(r[te]['macro_f1'] - old[te]['macro_f1']) for te in ('iot', 'ent')); dmax.append(d_)
        print(f'  FA-MLP train={tr:<6} s{s:<5} IoT {r["iot"]["macro_f1"]:.4f} | Ent {r["ent"]["macro_f1"]:.4f} | |diff vs Cell 5| {d_:.5f}'
              f' | {(time.time() - t0) / 60:.0f} min', flush=True)
print(f'Step 1 reproduction: max |diff| = {max(dmax):.5f} over {len(dmax)} runs')

# ---- Step 2: classical baselines, fixed settings (same as the Computer Networks benchmark) ----
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
try: import xgboost as xgb
except ImportError: os.system('pip -q install xgboost'); import xgboost as xgb
try: import lightgbm as lgb
except ImportError: os.system('pip -q install lightgbm'); import lightgbm as lgb
BASE = {'LogReg': lambda: LogisticRegression(max_iter=2000, class_weight='balanced', n_jobs=-1),
        'GaussianNB': lambda: GaussianNB(),
        'kNN': lambda: KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        'DecisionTree': lambda: DecisionTreeClassifier(class_weight='balanced', random_state=42),
        'RandomForest': lambda: RandomForestClassifier(n_estimators=200, class_weight='balanced', n_jobs=-1, random_state=42),
        'XGBoost': lambda: xgb.XGBClassifier(n_estimators=300, learning_rate=0.1, max_depth=8, tree_method='hist', device='cuda', random_state=42),
        'LightGBM': lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=63, class_weight='balanced',
                                               random_state=42, n_jobs=-1, verbose=-1)}
for sp in ('random', 'time'):
    DOM = {d: load_c5(d, sp) for d in ('iot', 'ent')}
    for tr, doms in SETS.items():
        Xtr = np.concatenate([DOM[d]['X_train'] for d in doms]); ytr = np.concatenate([DOM[d]['y_train'] for d in doms])
        for name, make in BASE.items():
            fp = f'{ST}/base_{sp}_{tr}_{name}_s42.json'
            if not os.path.exists(fp):
                mdl = make(); tt = time.time()
                if name == 'XGBoost': mdl.fit(Xtr, ytr, sample_weight=compute_sample_weight('balanced', ytr))
                else: mdl.fit(Xtr, ytr)
                res = {'fit_s': time.time() - tt}
                for te in ('iot', 'ent'):
                    p = np.asarray(mdl.predict(DOM[te]['X_test'])).astype(int)
                    res[te] = {'macro_f1': mf1(DOM[te]['y_test'], p), 'per_class': pcf(DOM[te]['y_test'], p), 'cm': cmx(DOM[te]['y_test'], p)}
                json.dump(res, open(fp, 'w')); del mdl; gc.collect()
            r = json.load(open(fp)); chk = ''
            if name == 'LightGBM':
                old = json.load(open(f'{XS}/central/{sp}_{tr}_lgbm_s42.json'))
                chk = f' | |diff vs Cell 5| {max(abs(r[te]["macro_f1"] - old[te]["macro_f1"]) for te in ("iot", "ent")):.5f}'
            print(f'  {sp:<6} train={tr:<6} {name:<12} IoT {r["iot"]["macro_f1"]:.4f} | Ent {r["ent"]["macro_f1"]:.4f}{chk}'
                  f' | {(time.time() - t0) / 60:.0f} min', flush=True)

# ---- Step 3: federated runs with round-100 confusion matrices (run() identical to Cell 12) ----
def run(DATA, setting, sp, alpha, method, seed, lr, with_test, outdir):
    kind = 'iid' if alpha is None else 'dirichlet'
    tag = f'{sp}_{kind}{alpha if alpha else ""}_{setting}_{method}_lr{lr}_s{seed}'
    fp = f'{outdir}/{tag}.json'
    if os.path.exists(fp): return json.load(open(fp)), True
    doms = ('iot', 'ent') if setting == 'xsilo' else (setting,)
    cat = lambda k: np.concatenate([DATA[d][k] for d in doms])
    X, y, Xv, yv = cat('X_train'), cat('y_train'), cat('X_val'), cat('y_val')
    Xt, yt = (cat('X_test'), cat('y_test')) if with_test else (None, None)
    parts, off = [], 0
    for d in doms:
        parts += [p + off for p in fl_xs.partition(DATA[d]['y_train'], kind, 10, seed, alpha=alpha)]
        off += len(DATA[d]['y_train'])
    cv = np.cumsum([0] + [len(DATA[d]['y_val']) for d in doms]); ct = np.cumsum([0] + [len(DATA[d]['y_test']) for d in doms])
    div = False
    try:
        o = fl_xs.run_fl(X, y, Xv, yv, Xt, yt, parts, method, seed, rounds=100, local_epochs=1, batch=256, lr=lr,
                         momentum=0.9, mu=0.01, dyn_alpha=0.001, eval_every=10, clip_norm=1.0)
    except FloatingPointError as e:
        o = {'history': []}; div = str(e)
    hist = []
    for h in o['history']:
        rec = {'round': h['round']}
        for i, d in enumerate(doms):
            rec[f'val_{d}'] = mf1(yv[cv[i]:cv[i + 1]], h['pv'][cv[i]:cv[i + 1]])
            if with_test:
                rec[f'test_{d}'] = mf1(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'test_pc_{d}'] = pcf(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]])
                rec[f'npred_{d}'] = int(len(np.unique(h['pt'][ct[i]:ct[i + 1]])))
                if h['round'] == 100: rec[f'cm_{d}'] = confusion_matrix(yt[ct[i]:ct[i + 1]], h['pt'][ct[i]:ct[i + 1]], labels=np.arange(NC)).tolist()
        hist.append(rec)
    res = {'tag': tag, 'setting': setting, 'split': sp, 'partition': f'{kind}{alpha if alpha else ""}', 'method': method,
           'seed': seed, 'lr': lr, 'diverged': div, 'minutes': o.get('minutes'), 'client_sizes': o.get('client_sizes'),
           'partition_sig': o.get('partition_sig'), 'init_sig': o.get('init_sig'), 'history': hist}
    json.dump(res, open(fp, 'w')); del o; gc.collect()
    return res, False
last3 = lambda res, key: float(np.mean([h[key] for h in res['history'] if h['round'] in (80, 90, 100)]))
FLD = {d: load_fl(d, 'random') for d in ('iot', 'ent')}
OUT = f'{RB}/D_confusion'; os.makedirs(OUT, exist_ok=True)          # same folder as Cell 12-D: its 3 FedAvg seeds are reused
JOBS = [(st, m, s) for m in ('fedavg', 'feddyn') for s in SEEDS for st in ('iot', 'ent', 'xsilo')]
EST = {'iot': 2.5, 'ent': 2.5, 'xsilo': 5.0}
fpj = lambda st, m, s: f'{OUT}/random_dirichlet0.5_{st}_{m}_lr{LR[st][m]}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fpj(*j))]
print(f'\nStep 3: {len(JOBS)} federated runs | {len(JOBS) - len(todo)} done | ~{sum(EST[j[0]] * (1.2 if j[1] == "feddyn" else 1) for j in todo) / 60:.1f} h left')
dfl = []
for i, (st, m, s) in enumerate(JOBS):
    r, cached = run(FLD, st, 'random', 0.5, m, s, LR[st][m], with_test=True, outdir=OUT)
    main = json.load(open(f'{XS}/grid/random_dirichlet0.5_{st}_{m}_lr{LR[st][m]}_s{s}.json'))
    doms = ('iot', 'ent') if st == 'xsilo' else (st,)
    d_ = max(abs(last3(r, f'test_{d}') - last3(main, f'test_{d}')) for d in doms); dfl.append(d_)
    if not cached:
        print(f'  [{i + 1}/{len(JOBS)}] {st:<5} {m:<7} s{s:<5} ' + ' | '.join(f'{d} {last3(r, "test_" + d):.4f}' for d in doms)
              + f' | |diff vs main grid| {d_:.5f} | {r["minutes"] or 0:.1f} min | {(time.time() - t0) / 60:.0f} min total', flush=True)
print(f'Step 3 reproduction: max |diff| vs main grid = {max(dfl):.5f} over {len(dfl)} runs')

# ---- Step 4 ----
export('export_xs_std.zip', ['stdmetrics/*.json', 'robust/D_confusion/*.json'])
print(f'CELL 13 DONE in {(time.time() - t0) / 60:.1f} min')

In [ ]:
# CROSS-SILO FL — CELL 14: partial personalization (reviewer check, EXPLORATORY). GPU T4, ~1.7 h (21 runs x ~5 min), resumable.
#   Question: if the IoT cost comes from interference in shared layers, does keeping some layers silo-local remove it?
#   Variants (FedAvg, cross-silo learning rate 0.3, random split, Dirichlet 0.5, the same 20 clients and seeds as the main grid):
#     local_head    : output layer averaged within each silo only (FedPer-style); all other layers shared by all 20 clients
#     local_encoder : feature-attention layer + first dense layer averaged within each silo; the rest shared
#   Step 0: the same loop with ALL layers shared must reproduce the main-grid cross-silo FedAvg run (seed 42) exactly.
#   Output: results/crosssilo/personal/*.json + export_xs_personal.zip (download). Prints relative paths only.
import os, glob, json, time, sys, importlib, gc, zipfile
import numpy as np, pandas as pd, warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import f1_score, confusion_matrix
from scipy import stats
from google.colab import drive, files
drive.mount('/content/drive')
cands = [os.path.dirname(os.path.dirname(p)) for p in glob.glob('/content/drive/MyDrive/*/data/raw_sampled.parquet')]
cands = [c for c in cands if os.path.isdir(f'{c}/results') and os.path.isdir(f'{c}/checkpoints')]
assert len(cands) == 1; DRIVE_PATH = cands[0]; D = f'{DRIVE_PATH}/data'; V2D = f'{D}/v2'; XSD = f'{D}/crosssilo'
XS = f'{DRIVE_PATH}/results/crosssilo'; PD = f'{XS}/personal'; os.makedirs(PD, exist_ok=True)
LR = json.load(open(f'{XS}/lr_chosen.json'))

src = open(f'{V2D}/fl_core.py').read(); assert src.count('N_CLASSES = 8') == 1
open('/content/fl_xp.py', 'w').write(src.replace('N_CLASSES = 8', 'N_CLASSES = 5')); sys.path.insert(0, '/content')
import fl_xp; importlib.reload(fl_xp)
import tensorflow as tf
tf.config.experimental.enable_op_determinism()
print('GPU:', tf.config.list_physical_devices('GPU'), '| head classes:', fl_xp.N_CLASSES)
FAM5 = ['Benign', 'Brute Force', 'DDoS', 'DoS', 'Recon']; NC = 5; LAB = np.arange(NC)
SEEDS = [42, 123, 456, 789, 1011, 2024, 7, 99, 314, 2718]
mf1 = lambda y, p: float(f1_score(y, p, average='macro', labels=LAB))
pcf = lambda y, p: [float(v) for v in f1_score(y, p, average=None, labels=LAB)]
def load(dom, sp):                     # loader of Cells 7-13, unchanged
    S = np.load(f'{XSD}/{dom}_{sp}.npz', allow_pickle=True); out = {}
    for k in ('train', 'val', 'test'):
        m = S[f'y_{k}'] < NC
        out[f'X_{k}'] = S[f'X_{k}'][m].astype(np.float32); out[f'y_{k}'] = S[f'y_{k}'][m].astype(np.int32)
    return out
DATA = {d: load(d, 'random') for d in ('iot', 'ent')}
XV = np.concatenate([DATA[d]['X_val'] for d in ('iot', 'ent')]); XT = np.concatenate([DATA[d]['X_test'] for d in ('iot', 'ent')])
CV = np.cumsum([0] + [len(DATA[d]['y_val']) for d in ('iot', 'ent')]); CT = np.cumsum([0] + [len(DATA[d]['y_test']) for d in ('iot', 'ent')])
VARIANTS = {'shared': set(), 'local_head': {6, 7}, 'local_encoder': {0, 1, 2, 3}}   # indices into the 8 trainable tensors

def run_pers(local, seed, lr, rounds=100, batch=256, momentum=0.9, clip_norm=1.0, eval_every=10):
    doms = ('iot', 'ent')
    X = np.concatenate([DATA[d]['X_train'] for d in doms]); y = np.concatenate([DATA[d]['y_train'] for d in doms])
    parts, silo, off = [], [], 0
    for i, d in enumerate(doms):                                   # identical clients to the main grid
        ps = fl_xp.partition(DATA[d]['y_train'], 'dirichlet', 10, seed, alpha=0.5)
        parts += [p + off for p in ps]; silo += [i] * len(ps); off += len(DATA[d]['y_train'])
    silo = np.array(silo)
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(seed)
    model = fl_xp.build_model(X.shape[1], True); tv = model.trainable_variables; assert len(tv) == 8
    K = len(parts); n_k = np.array([len(p) for p in parts], dtype=np.float64)
    gw = [w.numpy().copy() for w in tv]; init_sig = fl_xp.wsig(gw)
    sw = {i: [w.copy() for w in gw] for i in range(2)}             # model of each silo (shared tensors identical)
    buf = [tf.Variable(np.zeros_like(w), trainable=False) for w in gw]
    lossf = tf.keras.losses.SparseCategoricalCrossentropy()
    lr_t, mom_t = tf.constant(lr, tf.float32), tf.constant(momentum, tf.float32)

    @tf.function(reduce_retracing=True)
    def local_epoch(Xc, yc, perm):
        nb = tf.shape(perm)[0] // batch
        for i in tf.range(nb):
            bi = perm[i * batch:(i + 1) * batch]; xb, yb = tf.gather(Xc, bi), tf.gather(yc, bi)
            with tf.GradientTape() as tape:
                loss = lossf(yb, model(xb, training=True))
            grads = tape.gradient(loss, tv)
            if clip_norm is not None: grads, _ = tf.clip_by_global_norm(grads, clip_norm)
            for w, b, g in zip(tv, buf, grads): b.assign(mom_t * b + g); w.assign_sub(lr_t * b)
        return nb

    gpu = {k: (tf.constant(X[p]), tf.constant(y[p].astype(np.int32))) for k, p in enumerate(parts)}
    hist = []; t0 = time.time()
    for r in range(1, rounds + 1):
        local_ws = []
        for k in range(K):
            for v, w in zip(tv, sw[silo[k]]): v.assign(w)
            for b in buf: b.assign(tf.zeros_like(b))
            Xc, yc = gpu[k]
            perm = tf.constant(np.random.default_rng(seed * 100003 + r * 101 + k * 7 + 0).permutation(int(n_k[k])), tf.int32)
            local_epoch(Xc, yc, perm); local_ws.append([v.numpy().copy() for v in tv])
        a = n_k / n_k.sum()
        shared = [sum(a[k] * local_ws[k][L] for k in range(K)) for L in range(8)]          # identical to fl_core FedAvg
        for i in range(2):
            ai = np.where(silo == i, n_k, 0.0); ai = ai / ai.sum()
            loc = [sum(ai[k] * local_ws[k][L] for k in range(K)) for L in range(8)] if local else None
            sw[i] = [loc[L] if L in local else shared[L] for L in range(8)]
        if r == 1 or r % eval_every == 0 or r == rounds:
            rec = {'round': r}
            for i, d in enumerate(doms):                    # predict on the concatenated sets exactly as Cell 7 did, keep silo i's slice
                for v, w in zip(tv, sw[i]): v.assign(w)
                pv = model.predict(XV, batch_size=8192, verbose=0).argmax(1)[CV[i]:CV[i + 1]]
                pt = model.predict(XT, batch_size=8192, verbose=0).argmax(1)[CT[i]:CT[i + 1]]
                rec[f'val_{d}'] = mf1(DATA[d]['y_val'], pv); rec[f'test_{d}'] = mf1(DATA[d]['y_test'], pt)
                rec[f'test_pc_{d}'] = pcf(DATA[d]['y_test'], pt); rec[f'npred_{d}'] = int(len(np.unique(pt)))
                if r == rounds: rec[f'cm_{d}'] = confusion_matrix(DATA[d]['y_test'], pt, labels=LAB).tolist()
            hist.append(rec)
            if not all(np.isfinite(w).all() for i in range(2) for w in sw[i]): return {'diverged': True, 'history': hist}
    del gpu, model; tf.keras.backend.clear_session(); gc.collect()
    return {'diverged': False, 'history': hist, 'minutes': (time.time() - t0) / 60, 'init_sig': init_sig,
            'partition_sig': fl_xp.partition_sig(parts), 'client_sizes': n_k.astype(int).tolist(), 'local_tensors': sorted(local), 'lr': lr, 'seed': seed}
L3 = lambda o, k: float(np.mean([h[k] for h in o['history'] if h['round'] in (80, 90, 100)]))
g = lambda st, m, s, lr: json.load(open(f'{XS}/grid/random_dirichlet0.5_{st}_{m}_lr{lr}_s{s}.json'))
LRX = LR['xsilo']['fedavg']

# ---- Step 0: all layers shared must reproduce the main-grid cross-silo FedAvg run, seed 42 ----
fp0 = f'{PD}/check_shared_fedavg_s42.json'
if not os.path.exists(fp0): json.dump(run_pers(set(), 42, LRX), open(fp0, 'w'))
o, ref = json.load(open(fp0)), g('xsilo', 'fedavg', 42, LRX)
d0 = max(abs(a[f'{k}_{d}'] - b[f'{k}_{d}']) for a, b in zip(o['history'], ref['history']) for d in ('iot', 'ent') for k in ('val', 'test'))
same = (len(o['history']) == len(ref['history']), o['client_sizes'] == ref['client_sizes'], o['partition_sig'] == ref['partition_sig'], o['init_sig'] == ref['init_sig'])
print(f'Step 0 reproduction (all layers shared vs main grid, seed 42, all 11 evaluations): max |diff| = {d0:.6f} | rounds/clients/partition/init identical: {same}')
assert d0 < 1e-3 and all(same), 'loop does not reproduce the main grid - STOP and paste this output to Claude'

# ---- Step 1: personalization variants ----
JOBS = [(v, s) for v in ('local_head', 'local_encoder') for s in SEEDS]
fp = lambda v, s: f'{PD}/{v}_fedavg_lr{LRX}_s{s}.json'
todo = [j for j in JOBS if not os.path.exists(fp(*j))]; t0 = time.time()
print(f'{len(JOBS)} runs | {len(JOBS) - len(todo)} done | ~{len(todo) * 5.0 / 60:.1f} h left')
for i, (v, s) in enumerate(todo):
    o = run_pers(VARIANTS[v], s, LRX); o['variant'] = v; json.dump(o, open(fp(v, s), 'w'))
    print(f'  [{i + 1}/{len(todo)}] {v:<14} s{s:<5} IoT {L3(o, "test_iot"):.4f} | Ent {L3(o, "test_ent"):.4f}' + (' DIVERGED' if o['diverged'] else '')
          + f' | {o.get("minutes", 0):.1f} min | ~{(len(todo) - i - 1) * 5.0 / 60:.1f} h left', flush=True)

# ---- summary: paired over seeds, pp ----
def pr(a, b, lab):
    d = 100 * (np.array(b) - np.array(a)); se = d.std(ddof=1) / np.sqrt(len(d)); t = stats.t.ppf(.975, len(d) - 1)
    return f'{lab}: {np.mean(a):.4f} -> {np.mean(b):.4f} = {d.mean():+.2f} pp [{d.mean() - t * se:+.2f}, {d.mean() + t * se:+.2f}] (neg {int((d < 0).sum())}/{len(d)}, p {stats.ttest_rel(b, a).pvalue:.4f})'
print('\n===== random split, Dirichlet 0.5, FedAvg, 10 seeds =====')
for v in ('local_head', 'local_encoder'):
    for d in ('iot', 'ent'):
        pv = [L3(json.load(open(fp(v, s))), f'test_{d}') for s in SEEDS]
        w1 = [L3(g(d, 'fedavg', s, LR[d]['fedavg']), f'test_{d}') for s in SEEDS]
        w3 = [L3(g(d, 'fedavg', s, 0.3), f'test_{d}') for s in SEEDS]
        xs = [L3(g('xsilo', 'fedavg', s, LRX), f'test_{d}') for s in SEEDS]
        print(f'  {v:<14} {d}: ' + pr(w1, pv, 'vs within (main lr)') + '\n' + ' ' * 22 + pr(w3, pv, 'vs within (lr 0.3)') + '\n' + ' ' * 22 + pr(xs, pv, 'vs cross-silo shared'))
zp = f'{XS}/export_xs_personal.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f'{PD}/*.json')): z.write(f, os.path.relpath(f, XS))
print(f'\nexported {len(glob.glob(f"{PD}/*.json"))} files -> results/crosssilo/export_xs_personal.zip'); files.download(zp)
print(f'CELL 14 DONE in {(time.time() - t0) / 60:.1f} min')
